In [ ]:
# make two plots of the same spherical harmonic
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm, colors
import numpy as np
import matplotlib.pyplot as plt
try:
    from scipy.special import sph_harm
except ImportError:  # SciPy >= 1.17 removed sph_harm; same function, new argument order
    from scipy.special import sph_harm_y
    def sph_harm(m, l, phi, theta):
        return sph_harm_y(l, m, theta, phi)
from scipy.special import eval_genlaguerre
from scipy.special import lpmv
from scipy.special import factorial
%matplotlib inline
from scipy.optimize import root


plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif', 'serif']
plt.rcParams['font.family'] = 'serif'
plt.rcParams['mathtext.fontset'] = 'cm'

# Updated colormap access to avoid deprecation warnings
orig_map = plt.colormaps['seismic']
reversed_map = orig_map.reversed()

a0 = 1.0 # radial unit of Bohr!

def hydrogen_atom_wf(r,theta,phi,n,l,m):
    Y_norm = np.sqrt((2*l+1)*factorial(l-np.abs(m))/(4*np.pi*factorial(l+np.abs(m))))
    R_prefactor = -np.sqrt(factorial(n-l-1)/(2*n*factorial(n+l)))*(2.0/(n*a0))**(l+1.5)*np.power(r,l)*np.exp(-r/(n*a0))
    R = R_prefactor*eval_genlaguerre(n-l-1,2*l+1,2*r/(n*a0))
    return Y_norm*sph_harm(m, l, phi, theta).real*R

def hydrogen_atom_wf_imag(r,theta,phi,n,l,m):
    Y_norm = np.sqrt((2*l+1)*factorial(l-np.abs(m))/(4*np.pi*factorial(l+np.abs(m))))
    R_prefactor = -np.sqrt(factorial(n-l-1)/(2*n*factorial(n+l)))*(2.0/(n*a0))**(l+1.5)*np.power(r,l)*np.exp(-r/(n*a0))
    R = R_prefactor*eval_genlaguerre(n-l-1,2*l+1,2*r/(n*a0))
    return Y_norm*sph_harm(m, l, phi, theta).imag*R

def plot_hydrogen_atom_wf(n,l,m, ax_obj, r=np.linspace(0,10,100), theta=np.linspace(0,np.pi,20), phi=np.linspace(0,1.5*np.pi,25)):
    R, THETA, PHI = np.meshgrid(r, theta, phi)
    R = R.flatten()
    THETA = THETA.flatten()
    PHI = PHI.flatten()
    x = R*np.sin(THETA)*np.cos(PHI)
    y = R*np.sin(THETA)*np.sin(PHI)
    z = R*np.cos(THETA)
    wf = hydrogen_atom_wf(R,THETA,PHI,n,l,m)
    vmax = max(np.abs(np.amin(wf)),np.abs(np.amax(wf)))
    vmin = -vmax
    # Split title: Large orbital symbol and smaller text description
    ax_obj.set_title(rf'$\psi_{{{n},{l},{m}}}$', fontsize=60, y=-0.25)
    ax_obj.text2D(0.5, -0.3, '(Real)', transform=ax_obj.transAxes, ha='center', va='top', fontsize=40)

    ax_obj.scatter3D(x,y,z,c=wf, cmap=reversed_map, vmin=vmin, vmax=vmax,alpha=0.25)
    ax_obj.set_box_aspect((1,1,1))
    # Axis label sizes and padding (20 as requested)
    ax_obj.set_xlabel('$x$', fontsize=30, labelpad=15)
    ax_obj.set_ylabel('$y$', fontsize=30, labelpad=15)
    ax_obj.set_zlabel('$z$', fontsize=30, labelpad=15)
    ax_obj.tick_params(axis='both', which='major', labelsize=22)

In [ ]:
import matplotlib.pyplot as plt

# Define the rows manually for the 3D layout
rows = [
    [(1, 0, 0)],
    [(2, 0, 0), (2, 1, -1), (2, 1, 0), (2, 1, 1)],
    [(3, 0, 0), (3, 1, -1), (3, 1, 0), (3, 1, 1)],
    [(3, 2, -2), (3, 2, -1), (3, 2, 0), (3, 2, 1), (3, 2, 2)]
]

max_cols = 5
# Increased figsize to accommodate larger labels
fig = plt.figure(figsize=(36, 36), facecolor='w')

for i, row_orbitals in enumerate(rows):
    for j, (n, l, m) in enumerate(row_orbitals):
        # Add a 3D subplot at the correct position in the 4x5 grid
        ax = fig.add_subplot(len(rows), max_cols, i * max_cols + j + 1, projection='3d')
        plot_hydrogen_atom_wf(n, l, m, ax)

plt.tight_layout(pad=6.0)
plt.show();

In [ ]:
def plot_hydrogen_atom_wf_imag(n, l, m, ax_obj, r=np.linspace(0,10,100), theta=np.linspace(0,np.pi,20), phi=np.linspace(0,1.5*np.pi,25)):
    R, THETA, PHI = np.meshgrid(r, theta, phi)
    R, THETA, PHI = R.flatten(), THETA.flatten(), PHI.flatten()
    x = R * np.sin(THETA) * np.cos(PHI)
    y = R * np.sin(THETA) * np.sin(PHI)
    z = R * np.cos(THETA)
    wf = hydrogen_atom_wf_imag(R, THETA, PHI, n, l, m)

    vmax = max(np.abs(np.amin(wf)), np.abs(np.amax(wf)))

    # Always set the orbital title and description
    ax_obj.set_title(rf'$\psi_{{{n},{l},{m}}}$', fontsize=60, y=-0.25)
    ax_obj.text2D(0.5, -0.3, '(Imaginary)', transform=ax_obj.transAxes, ha='center', va='top', fontsize=40)

    # If the imaginary part is essentially zero, hide the axes/box and return
    if vmax < 1e-12:
        ax_obj.set_axis_off()
        return

    vmin = -vmax
    orig_map = plt.colormaps['seismic']
    reversed_map = orig_map.reversed()

    ax_obj.scatter3D(x, y, z, c=wf, cmap=reversed_map, vmin=vmin, vmax=vmax, alpha=0.25)
    ax_obj.set_box_aspect((1,1,1))
    # Consistent labelpad of 15
    ax_obj.set_xlabel('$x$', fontsize=30, labelpad=15)
    ax_obj.set_ylabel('$y$', fontsize=30, labelpad=15)
    ax_obj.set_zlabel('$z$', fontsize=30, labelpad=15)
    ax_obj.tick_params(axis='both', which='major', labelsize=22)

In [ ]:
rows = [
    [(1, 0, 0)],
    [(2, 0, 0), (2, 1, -1), (2, 1, 0), (2, 1, 1)],
    [(3, 0, 0), (3, 1, -1), (3, 1, 0), (3, 1, 1)],
    [(3, 2, -2), (3, 2, -1), (3, 2, 0), (3, 2, 1), (3, 2, 2)]
]

fig = plt.figure(figsize=(36, 36), facecolor='w')
for i, row_orbitals in enumerate(rows):
    for j, (n, l, m) in enumerate(row_orbitals):
        ax = fig.add_subplot(len(rows), 5, i * 5 + j + 1, projection='3d')
        plot_hydrogen_atom_wf_imag(n, l, m, ax)

plt.tight_layout(pad=6.0)
plt.show()

# xy-projection

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_hydrogen_atom_wf_xy_projection(n, l, m, ax_obj):
    # Grid in the XY plane (z=0)
    limit = 10
    x = np.linspace(-limit, limit, 1000)
    y = np.linspace(-limit, limit, 1000)
    X, Y = np.meshgrid(x, y)
    Z = np.zeros(X.shape)

    R = np.sqrt(X**2 + Y**2 + Z**2).flatten()
    # Avoid division by zero at origin
    R_safe = np.where(R == 0, 1e-15, R)
    THETA = np.arccos(Z.flatten() / R_safe)
    PHI = np.arctan2(Y.flatten(), X.flatten())

    # Use the hydrogen_atom_wf function defined in earlier cells
    wf = hydrogen_atom_wf(R, THETA, PHI, n, l, m)
    wf = wf.reshape(X.shape)

    vmax = max(np.abs(np.amin(wf)), np.abs(np.amax(wf)))
    if vmax == 0: vmax = 1.0 # Handle nodal plane cases to avoid errors
    vmin = -vmax

    # Apply requested styling: large fonts and title at the bottom
    ax_obj.set_title(rf'$\psi_{{{n},{l},{m}}}$', fontsize=40, y=-0.35)
    c = ax_obj.pcolormesh(X, Y, wf, cmap='RdBu', vmin=vmin, vmax=vmax)
    ax_obj.set_xlabel('$x$', fontsize=30)
    ax_obj.set_ylabel('$y$', fontsize=30)
    ax_obj.tick_params(axis='both', which='major', labelsize=22)
    ax_obj.axis([-limit, limit, -limit, limit])
    ax_obj.set_aspect('equal')
    return c

# Define the rows manually to satisfy the 4-row layout request
rows = [
    [(1, 0, 0)],
    [(2, 0, 0), (2, 1, -1), (2, 1, 0), (2, 1, 1)],
    [(3, 0, 0), (3, 1, -1), (3, 1, 0), (3, 1, 1)],
    [(3, 2, -2), (3, 2, -1), (3, 2, 0), (3, 2, 1), (3, 2, 2)]
]

max_cols = 5
fig, axes = plt.subplots(len(rows), max_cols, figsize=(32, 28), facecolor='w')

for i, row_orbitals in enumerate(rows):
    for j, (n, l, m) in enumerate(row_orbitals):
        plot_hydrogen_atom_wf_xy_projection(n, l, m, axes[i, j])

    # Hide axes for unused columns in the current row
    for j in range(len(row_orbitals), max_cols):
        axes[i, j].set_axis_off()

plt.tight_layout(pad=4.0)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

# Set font styles consistent with previous plots
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif', 'serif']
plt.rcParams['font.family'] = 'serif'
plt.rcParams['mathtext.fontset'] = 'cm'

# Define the principal quantum numbers n for the hydrogen atom
n_max = 5
n_values = np.arange(1, n_max + 1)
# Energy in atomic units (Hartrees): E_n = -0.5 / n^2
energies = -0.5 / (n_values**2)

# Figure setup
fig, ax = plt.subplots(figsize=(16, 12))

# Geometry for the state lines
base_segment_width = 0.2
segment_gap = 0.01
start_x_fixed = 0.1  # Fixed starting position for left alignment

# Color coding for l values
l_colors = {0: 'red', 1: 'green', 2: 'blue', 3: 'orange', 4: 'purple'}

# Calculate maximum width based on n=5 (25 states) to keep scaling consistent
max_states_to_plot = 5**2
max_width_n5 = max_states_to_plot * base_segment_width + (max_states_to_plot - 1) * segment_gap

for n, e in zip(n_values, energies):
    # Show full degeneracy for n <= 5, only 1 state for n > 5
    if n <= 5:
        # Increase the horizontal width of segments for n=1, 2, 3 else base_segment_width
        current_sw = base_segment_width * 2.5 if n <= 3 else base_segment_width
        lw = 4

        state_idx = 0
        for l in range(n):
            color = l_colors.get(l, 'gray')
            for m_l in range(-l, l + 1):
                x_s = start_x_fixed + state_idx * (current_sw + segment_gap)
                x_e = x_s + current_sw

                ax.hlines(e, x_s, x_e, colors=color, linewidth=lw)

                # Label m_l below each segment only for n <= 3
                if n <= 3:
                    ax.text((x_s + x_e)/2, e - 0.01, rf'$m_ℓ={m_l}$', ha='center', va='top', fontsize=14, color=color, alpha=0.8)

                state_idx += 1

        # Label the principal quantum number n
        ax.text(start_x_fixed + state_idx * (current_sw + segment_gap) + 0.02, e, rf'$n={n}$', va='center', fontsize=20)

    else:
        # Stop showing degeneracy for n > 5
        current_sw = base_segment_width
        lw = 4
        ax.hlines(e, start_x_fixed, start_x_fixed + current_sw, colors='black', linewidth=lw)

# Add E=0 line (Ionization limit) as a straight (solid) black line
ax.axhline(0, color='black', linestyle='-', linewidth=2)
# Moved the label to the left side (x=start_x_fixed)
ax.text(start_x_fixed, 0, 'Ionization Limit ($E=0$)', va='bottom', fontsize=20, color='black')

# Set y-axis ticks only for levels where labels don't overlap
ax.set_yticks(energies[:5])
# Show fractional values in atomic units (E_n = -1 / 2n^2)
y_tick_labels = [rf"$- \frac{{1}}{{{2*n**2}}}$" for n in n_values[:5]]
ax.set_yticklabels(y_tick_labels, fontsize=20)

ax.set_ylabel(r'Energy ($E$/a.u.)', fontsize=22)
# Adjust x-limits to accommodate labels
ax.set_xlim(start_x_fixed - 0.5, start_x_fixed + max_width_n5 + 1.2)
# Rescale y-limits for atomic units
ax.set_ylim(-0.55, 0.05)
ax.set_xticks([])

# Add a grid for the energy axis
ax.yaxis.grid(True, linestyle='--', alpha=0.3)

#plt.title('Hydrogen Atom Energy Levels (Atomic Units)', fontsize=24)
plt.subplots_adjust(left=0.15, right=0.9, top=0.9, bottom=0.1)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import genlaguerre, factorial

def radial_wavefunction(n, l, r, a0=1.0):
    """
    Calculates the radial part of the hydrogen atom wavefunction R_nl(r).

    Args:
        n (int): Principal quantum number
        l (int): Azimuthal quantum number
        r (array): Radial distance from the nucleus
        a0 (float): Bohr radius (default 1.0 for atomic units)

    Returns:
        R_nl (array): The radial wavefunction
    """
    rho = (2 * r) / (n * a0)

    # Normalization constant
    norm = np.sqrt((2 / (n * a0))**3 * (factorial(n - l - 1) / (2 * n * factorial(n + l))))

    # Associated Laguerre polynomial L_{n-l-1}^{2l+1}
    laguerre = genlaguerre(n - l - 1, 2 * l + 1)

    return norm * np.exp(-rho / 2) * (rho**l) * laguerre(rho)


# Define the radial distance range (in units of Bohr radius a0)
r_max = 15.0
r = np.linspace(0, r_max, 500)

print(f"Radial range set from 0 to {r_max} a0 with {len(r)} points.")

# Define radial distance range (in units of a0)
r = np.linspace(0, 20, 500)

# Define the quantum states (n, l) to plot
states = [(1, 0), (2, 0), (2, 1), (3, 0), (3, 1), (3, 2)]
orbital_map = {0: 's', 1: 'p', 2: 'd', 3: 'f'}

In [ ]:
plt.figure(figsize=(12, 8))
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif', 'serif']
plt.rcParams['font.family'] = 'serif'
plt.rcParams['mathtext.fontset'] = 'cm'

for n, l in states:
    R_nl_vals = radial_wavefunction(n, l, r)
    # Using LaTeX to italicize the orbital name in the legend
    label = rf'${n}{orbital_map[l]}$'
    plt.plot(r, R_nl_vals, label=label)

    # Calculate radial nodes: roots of the associated Laguerre polynomial
    if n - l - 1 > 0:
        poly = genlaguerre(n - l - 1, 2 * l + 1)
        rho_roots = poly.roots
        r_roots = rho_roots * n / 2.0

        # Plot roots without a legend label
        valid_roots = r_roots[r_roots <= 15]
        plt.scatter(valid_roots, np.zeros_like(valid_roots),
                    color='purple', zorder=5)

plt.axhline(0, color='black', linewidth=0.5, linestyle='--')
plt.xlabel('Radius $r/a_0$', fontsize=20)
plt.ylabel('$R_{n\\ell}(r)$', fontsize=20)
plt.xlim(0, 15)
plt.ylim(-0.12, 0.8)
plt.legend(fontsize=20)

# Increase the size of the axes numbers (tick labels)
plt.tick_params(axis='both', which='major', labelsize=16)

plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif', 'serif']
plt.rcParams['font.family'] = 'serif'
plt.rcParams['mathtext.fontset'] = 'cm'

for n, l in states:
    R_nl_vals = radial_wavefunction(n, l, r)
    prob_density_r = (r**2) * (R_nl_vals**2)
    # Using LaTeX to italicize the orbital name in the legend
    label = rf'${n}{orbital_map[l]}$'
    plt.plot(r, prob_density_r, label=label)

    # Mark radial nodes without a legend label
    if n - l - 1 > 0:
        poly = genlaguerre(n - l - 1, 2 * l + 1)
        rho_roots = poly.roots
        r_roots = rho_roots * n / 2.0

        valid_roots = r_roots[r_roots <= 15]
        plt.scatter(valid_roots, np.zeros_like(valid_roots),
                    color='purple', zorder=5)

#plt.title('Radial Probability Density $r^2|R_{nl}(r)|^2$ with Nodes', fontsize=20)
plt.xlabel('Radius $r/a_0$', fontsize=20)
plt.ylabel('Probability Density $r^2|R_{n\\ell}(r)|^2$', fontsize=20)
plt.xlim(0, 15)
plt.legend(fontsize=20)

# Increase the size of the axes numbers (tick labels)
plt.tick_params(axis='both', which='major', labelsize=16)

plt.grid(True, alpha=0.3)
plt.show()